In [1]:
import sys
from pathlib import Path

import numpy as np
import polars as pl

import matplotlib.pyplot as plt

%load_ext autoreload
%autoreload 2

sys.path.insert(0, "..")
import plasmidtools


Paths

In [2]:
DATA_DIR = Path("../data")
FIG_DIR = DATA_DIR / "figures"
ADDGENE_DIR = DATA_DIR / "addgene"


Pre-processed / pre-calculated data

In [3]:
plasmid_meta = pl.read_csv(ADDGENE_DIR / "mammalian_plasmids.tsv", separator="\t")
plasmid_citations = pl.read_parquet(ADDGENE_DIR / "citations_addgene.parquet")
plasmid_stats = pl.read_parquet(ADDGENE_DIR / "mammalian_plasmids_statistics.parquet")

element_positions = pl.read_parquet(ADDGENE_DIR / "mammalian_plasmids_elements.parquet")
element_citations = pl.read_parquet(ADDGENE_DIR / "citations_addgene_elements.parquet")
primer_positions = pl.read_parquet(ADDGENE_DIR / "mammalian_plasmids_primers.parquet")
primer_citations = pl.read_parquet(ADDGENE_DIR / "citations_addgene_primers.parquet")

cre_annotation = pl.read_parquet(ADDGENE_DIR / "mammalian_plasmids_cre_and_tss.parquet")

element_cre_overlap = (
    pl.read_parquet(ADDGENE_DIR / "mammalian_plasmids_element_cre_overlaps.parquet")
    .join(element_citations[["element_type", "element_name", "n_plasmids", "n_citations"]], left_on=["type", "name"], right_on=["element_type", "element_name"], how="left")
    .with_columns(pl.max_horizontal("tss_fwd_avg_signal", "tss_rev_avg_signal").alias("tss_avg_signal"))
)

primer_cre_overlap = (
    pl.read_parquet(ADDGENE_DIR / "mammalian_plasmids_primers_cre_overlaps.parquet")
    .join(primer_citations[["element_type", "element_name", "n_plasmids", "n_citations"]], left_on=["type", "name"], right_on=["element_type", "element_name"], how="left")
    .with_columns(pl.max_horizontal("tss_fwd_avg_signal", "tss_rev_avg_signal").alias("tss_avg_signal"))
)


Inserts table

In [95]:
plasmid_data = (
    plasmid_meta
    .join(plasmid_citations[["plasmid_id", "n_citations"]], on="plasmid_id", how="left")
    [["plasmid_id", "sequence_id", "plasmid_name", "vector_type", "backbone", "inserts", "n_citations"]]
)


In [96]:
plasmid_data

plasmid_id,sequence_id,plasmid_name,vector_type,backbone,inserts,n_citations
i64,i64,str,str,str,str,i64
100035,192913,"""pMT3-RhoGC D380E""","""Mammalian Expression""","""pMT3""","""RhoGC D380E""",1
100050,194451,"""pAAV.CaMKIIa.ChETA(E123T/H134R…","""Mammalian Expression ||| AAV""","""pAAV""","""hChR2(E123T/H134R)""",2
100034,193102,"""pMT3-RhoGC D380N""","""Mammalian Expression""","""pMT3""","""RhoGC D380N""",1
100027,195074,"""pcDNA3-Antares2 c-myc""","""Mammalian Expression""","""pcDNA3""","""Antares2""",5
100091,197565,"""dCas9 plasmid""","""Mammalian Expression ||| CRISP…","""pcDNA3.3-TOPO""","""""",8
100015,192451,"""NsvBa""","""Mammalian Expression""","""pTracer-CMV2, IRES GFP""","""NsvBa""",1
100049,194448,"""pAAV.hSynap.ChETA(E123T/H134R)…","""Mammalian Expression ||| AAV""","""pAAV""","""hChR2(E123T/H134R)""",2
100025,194289,"""pcDNA3.2-DEST-MatryoshCaMP6s""","""Mammalian Expression""","""pcDNA3.2/V5-DEST""","""MatryoshCaMP6s""",1
100106,193262,"""pSN21""","""Mammalian Expression ||| Lenti…","""pLM-vexGFP-OCT4""","""vexGFP-P2A-OCT4 fused to a 'we…",1


In [98]:
pl.Config.set_tbl_rows(25)

inserts_auto = (
    element_positions
    .filter(pl.col("element_type") == "CDS")
    .join(plasmid_data, on="sequence_id")

    .sort(["insert_or_backbone", "n_citations"], descending=True)
    .group_by("element_name", maintain_order=True)
    .first()

    [["plasmid_id", "plasmid_name", "vector_type", "backbone", "inserts", "n_citations", "insert_or_backbone", "element_name", "length", "strand", "intervals"]]

    # .filter(pl.col("n_citations") >= 5)

    .with_columns(
        positions=pl.format(
                "[{}]",  # Wrap the final joined string in outer brackets
                pl.col("intervals").list.eval(
                    # Format inner lists: cast to String, join with commas, wrap in brackets
                    pl.format(
                        "[{}]", 
                        pl.element().cast(pl.List(pl.String)).list.join(", ")
                    )
                ).list.join(", ") # Join the formatted inner lists with commas
            )
    )
    .drop("intervals")
)

inserts_auto.write_csv(ADDGENE_DIR / "mammalian_plasmids_inserts_autoannotated.csv")


In [100]:
(
    inserts_auto
    # .filter(pl.col("insert_or_backbone") == "backbone")
    # [["plasmid_name", "backbone", "inserts", "element_name"]]

    # .head(5)
)

plasmid_id,plasmid_name,vector_type,backbone,inserts,n_citations,insert_or_backbone,element_name,length,strand,positions
i64,str,str,str,str,i64,str,str,i64,i64,str
52961,"""lentiCRISPR v2""","""Mammalian Expression ||| Lenti…","""Custom""","""Cas9 ||| Puromycin resistance""",3627,"""insert""","""Cas9""",4104,1,"""[[4492, 8596]]"""
52961,"""lentiCRISPR v2""","""Mammalian Expression ||| Lenti…","""Custom""","""Cas9 ||| Puromycin resistance""",3627,"""insert""","""PuroR""",597,1,"""[[8734, 9331]]"""
12253,"""pRSV-Rev""","""Mammalian Expression ||| Lenti…","""pRSV-Rev""","""Rev""",1234,"""insert""","""Rev""",351,-1,"""[[3233, 3584]]"""
12251,"""pMDLg/pRRE""","""Mammalian Expression ||| Lenti…","""pMD""","""HIV-1 GAG/POL""",1210,"""insert""","""HIV-1 gag""",1503,-1,"""[[6113, 7616]]"""
34879,"""pCMV(CAT)T7-SB100""","""Mammalian Expression""","""pCMV""","""SB100X transposase""",301,"""insert""","""SB100X""",1023,1,"""[[856, 1879]]"""
17608,"""pRK5-HA-Ubiquitin-WT""","""Mammalian Expression""","""pRK5-HA""","""Ubiquitin C""",278,"""insert""","""ubiquitin""",228,1,"""[[1018, 1246]]"""
14883,"""FUGW""","""Mammalian Expression ||| Lenti…","""HR'CS-G""","""flap-Ub promoter-GFP-WRE""",266,"""insert""","""EGFP""",717,1,"""[[3886, 3889], [3889, 3892], […"
61425,"""lenti dCAS-VP64_Blast""","""Mammalian Expression ||| Lenti…","""plenti""","""dCAS9(D10A, N863A)-VP64_2A_Bla…",229,"""insert""","""dCas9""",4101,1,"""[[9955, 9979], [9979, 9982], […"
61425,"""lenti dCAS-VP64_Blast""","""Mammalian Expression ||| Lenti…","""plenti""","""dCAS9(D10A, N863A)-VP64_2A_Bla…",229,"""insert""","""VP64""",150,1,"""[[70, 220]]"""


In [103]:
pl.Config.set_tbl_rows(25)

inserts_auto = (
    element_positions
    .filter(pl.col("element_type") == "CDS")
    .join(plasmid_data, on="sequence_id")

    .sort(["insert_or_backbone", "n_citations"], descending=True)
    # .group_by("element_name", maintain_order=True)
    # .first()

    [["plasmid_id", "plasmid_name", "vector_type", "backbone", "inserts", "n_citations", "insert_or_backbone", "element_name", "length", "strand", "intervals"]]

    # .filter(pl.col("n_citations") >= 5)

    .with_columns(
        positions=pl.format(
                "[{}]",  # Wrap the final joined string in outer brackets
                pl.col("intervals").list.eval(
                    # Format inner lists: cast to String, join with commas, wrap in brackets
                    pl.format(
                        "[{}]", 
                        pl.element().cast(pl.List(pl.String)).list.join(", ")
                    )
                ).list.join(", ") # Join the formatted inner lists with commas
            )
    )
    .drop("intervals")
    .filter(pl.col("insert_or_backbone") == "insert")
)

# inserts_auto["insert_or_backbone"].value_counts()
inserts_auto.write_csv(ADDGENE_DIR / "mammalian_plasmids_inserts_autoannotated.csv")


In [104]:
inserts_auto

plasmid_id,plasmid_name,vector_type,backbone,inserts,n_citations,insert_or_backbone,element_name,length,strand,positions
i64,str,str,str,str,i64,str,str,i64,i64,str
52961,"""lentiCRISPR v2""","""Mammalian Expression ||| Lenti…","""Custom""","""Cas9 ||| Puromycin resistance""",3627,"""insert""","""Cas9""",4104,1,"""[[4492, 8596]]"""
52961,"""lentiCRISPR v2""","""Mammalian Expression ||| Lenti…","""Custom""","""Cas9 ||| Puromycin resistance""",3627,"""insert""","""PuroR""",597,1,"""[[8734, 9331]]"""
52961,"""lentiCRISPR v2""","""Mammalian Expression ||| Lenti…","""Custom""","""Cas9 ||| Puromycin resistance""",3627,"""insert""","""Cas9""",4104,1,"""[[4492, 8596]]"""
48138,"""pSpCas9(BB)-2A-GFP (PX458)""","""Mammalian Expression ||| CRISP…","""PX458""","""hSpCas9""",2346,"""insert""","""Cas9""",4101,1,"""[[6013, 9288], [0, 826]]"""
48138,"""pSpCas9(BB)-2A-GFP (PX458)""","""Mammalian Expression ||| CRISP…","""PX458""","""hSpCas9""",2346,"""insert""","""Cas9""",4101,1,"""[[6013, 9288], [0, 826]]"""
42230,"""pX330-U6-Chimeric_BB-CBh-hSpCa…","""Mammalian Expression ||| CRISP…","""pUC ori vector""","""humanized S. pyogenes Cas9""",1936,"""insert""","""Cas9""",4101,1,"""[[1373, 5474]]"""
42230,"""pX330-U6-Chimeric_BB-CBh-hSpCa…","""Mammalian Expression ||| CRISP…","""pUC ori vector""","""humanized S. pyogenes Cas9""",1936,"""insert""","""Cas9""",4101,1,"""[[1373, 5474]]"""
62988,"""pSpCas9(BB)-2A-Puro (PX459) V2…","""Mammalian Expression ||| CRISP…","""PX459""","""hSpCas9-2A-Puro V2.0""",1738,"""insert""","""PuroR""",600,1,"""[[5145, 5745]]"""
62988,"""pSpCas9(BB)-2A-Puro (PX459) V2…","""Mammalian Expression ||| CRISP…","""PX459""","""hSpCas9-2A-Puro V2.0""",1738,"""insert""","""Cas9""",4101,1,"""[[927, 5028]]"""
